# Large Langugae Model

A language model is a probability distribution over sequences of a vocabulary.

Let V be a vocabulary, items could be characters, tokens or words. A language model assigns a probability to each sequence of items. 

Let $(w_1, w_2, ... ,w_{n-1})$ be a sequence of words from vocabulary V and a LM a language model over V with probablity P then:

$$ P(w_n|w_1, ... ,w_{n-1}) = neuralnetwork(w_1, ... ,w_{n-1})$$

The model predicts the next word based on the history.

We would like a model to assign higher probabilities to sentences that are real and syntactically correct.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
######################## Load GPT2 from huggingface ############
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from transformers import AutoTokenizer, AutoModelForCausalLM

#llama3_tokenizer = AutoTokenizer.from_pretrained("NousResearch/Hermes-2-Pro-Llama-3-8B")
#llama3 = AutoModelForCausalLM.from_pretrained("NousResearch/Hermes-2-Pro-Llama-3-8B")
tokenizer = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2")

In [ ]:
print(model)
########## Let have a look at the vocabulary of GPT-2 ##########
vocab = tokenizer.get_vocab()
print(type(vocab))
print(len(vocab))

In [ ]:
text = "The theory of relativity usually encompasses two interrelated theories by Albert"
encoded_text = tokenizer(text, return_tensors="pt")
print(encoded_text)

In [ ]:
#1. Run the Model
with torch.inference_mode():
  outputs = model(**encoded_text)

In [ ]:
# print(outputs)

In [ ]:
print("1. step to get the logits of the next token")
loss, logits = outputs.loss, outputs.logits
next_token_logits = logits[0, -1, :] # batch_size, sequence_length, config.vocab_size)
print(f"logits is a {type(logits)} of shape {logits.shape} with a vocabulary size of {next_token_logits.shape}")

In [ ]:
print(next_token_logits)

In [ ]:
print("2. step to convert the logits to probabilities")
next_token_probs = torch.softmax(next_token_logits, -1)
print(next_token_probs.shape)
print(next_token_probs) # for each token in the vocabulary we get a probability

In [ ]:
print("3. step to get the top 10 and put all together")
topk_next_tokens= torch.topk(next_token_probs, 10) 
topk_next_token_list = [(tokenizer.decode(idx), prob) for idx, prob in zip(topk_next_tokens.indices, topk_next_tokens.values)] 
for token, prob in topk_next_token_list:
    print(round(prob.item(),3),"%", token)


In [ ]:
# get the first one
next_token, next_prob = topk_next_token_list[0]
print(next_token, next_prob)

In [ ]:
############################## Llama 3 ###################

In [ ]:
# Load model directly
# https://huggingface.co/NousResearch/Hermes-2-Pro-Llama-3-8B
from transformers import AutoTokenizer, AutoModelForCausalLM

llama3_tokenizer = AutoTokenizer.from_pretrained("NousResearch/Hermes-2-Pro-Llama-3-8B")
llama3 = AutoModelForCausalLM.from_pretrained("NousResearch/Hermes-2-Pro-Llama-3-8B")

In [ ]:
text = "The theory of relativity usually encompasses two interrelated theories by Albert"
encoded_text = llama3_tokenizer(text, return_tensors="pt")
print(encoded_text)

In [ ]:
#1. Run the Model
with torch.no_grad():
  outputs = llama3(**encoded_text)

In [ ]:
print("1. step to get the logits of the next token")
loss, logits = outputs.loss, outputs.logits
next_token_logits = logits[0, -1, :] # batch_size, sequence_length, config.vocab_size)
print(f"logits is a {type(logits)} of shape {logits.shape} with a vocabulary size of {next_token_logits.shape}")

In [ ]:
print("2. step to convert the logits to probabilities")
next_token_probs = torch.softmax(next_token_logits, -1)
print(next_token_probs.shape)
print(next_token_probs) # for each token in the vocabulary we get a probability

In [ ]:
print("3. step to get the top 10 and put all together")
topk_next_tokens= torch.topk(next_token_probs, 10) 
topk_next_token_list = [(llama3_tokenizer.decode(idx), prob) for idx, prob in zip(topk_next_tokens.indices, topk_next_tokens.values)] 
for token, prob in topk_next_token_list:
    print(round(prob.item(),3),"%", token)